In [17]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.applications import VGG19
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.preprocessing import image

## Build a Model with Dropout (Trainable=False, but Dropout Active)

In [18]:
def build_dropout_model(input_shape=(224, 224, 3), dropout_rate=0.4, num_classes=4):
    inputs = Input(shape=input_shape)
    base_model = VGG19(include_top=False, weights='imagenet', input_tensor=inputs)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(dropout_rate)(x, training=True)  # Important: keep dropout ON at inference
    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [19]:
def load_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    return img_array

## Predict Multiple Times

In [20]:
def predict_with_uncertainty(model, img_array, n_iter=50):
    predictions = np.array([model(img_array, training=True).numpy()[0] for _ in range(n_iter)])
    mean_pred = np.mean(predictions, axis=0)
    std_pred = np.std(predictions, axis=0)
    return mean_pred, std_pred

## trying with sample image 

In [21]:
# Load model
model = build_dropout_model(num_classes=4)  # Or 3 if you're using only pneumonia classes

# Load any test image
img_path = "Curated_X-Ray_Dataset/Pneumonia-Viral/Pneumonia-Viral (5).jpg"  # update this
img_array = load_image(img_path)

# Predict with uncertainty
mean_pred, std_pred = predict_with_uncertainty(model, img_array, n_iter=30)

# Show results
classes = ["COVID", "Bacterial", "Viral", "Normal"]
for i in range(len(mean_pred)):
    print(f"{classes[i]}: {mean_pred[i]*100:.2f}% ± {std_pred[i]*100:.2f}%")


COVID: 44.38% ± 44.92%
Bacterial: 0.00% ± 0.00%
Viral: 7.72% ± 21.57%
Normal: 47.90% ± 45.25%
